# Introduction to Biostatistics - Modelling: Logistic Regression

**Part 6: Modeling**

## Logistic Regression

We study the **prostate cancer dataset** from the Vanderbilt Biostatistics Library.

- **Reference:** Byar DP, Green SB (1980). *Bulletin Cancer*, Paris 67:477–488
- **Info:** [prostate](https://hbiostat.org/data/repo/prostate)
- **Data dictionary:** [cprostate](https://hbiostat.org/data/repo/cprostate)

The `status` variable is cleaned to create a binary `death` indicator (1 = dead, 0 = alive).

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

np.random.seed(1)

### Load and prepare data

In [2]:
data_prostate = pd.read_csv(
    'https://raw.githubusercontent.com/Decision-Academy/'
    'Intro-to-Biostatistics-with-Justin-Belair/refs/heads/main/data/prostate.csv'
)

# Extract the word before the first '-' in status, then create binary death indicator
data_prostate['death'] = (
    data_prostate['status']
    .str.extract(r'^([^-]*)')[0]
    .str.strip()
    .apply(lambda x: 1 if x == 'dead' else 0)
)

data_prostate.head()

,patno,stage,rx,dtime,status,age,wt,pf,hx,sbp,dbp,ekg,hg,sz,sg,ap,bm,sdate,death
0,1,3,0.2 mg estrogen,72,alive,75.0,76.0,normal activity,0,15,9,heart strain,13.798828,2.0,8.0,0.299988,0,2778,0
1,2,3,0.2 mg estrogen,1,dead - other ca,54.0,116.0,normal activity,0,13,7,heart block or conduction def,14.599609,42.0,NaN,0.699951,0,2820,1
2,3,3,5.0 mg estrogen,40,dead - cerebrovascular,69.0,102.0,normal activity,1,14,8,heart strain,13.398438,3.0,9.0,0.299988,0,2933,1
3,4,3,0.2 mg estrogen,20,dead - cerebrovascular,75.0,94.0,in bed < 50% daytime,1,14,7,benign,17.597656,4.0,8.0,0.899902,0,2999,1
4,5,3,placebo,65,alive,67.0,99.0,normal activity,0,17,10,normal,13.398438,34.0,8.0,0.500000,0,3002,0


### Fit a logistic regression model

Model death as a function of age, weight (`wt`), cancer stage, and treatment (`rx`).
We use the binomial family with a logit link function.

In [3]:
glm_logistic = smf.glm(
    'death ~ age + wt + stage + C(rx)',
    data=data_prostate,
    family=sm.families.Binomial(link=sm.families.links.Logit())
)

result = glm_logistic.fit()
print(result.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                  death   No. Observations:                  499
Model:                            GLM   Df Residuals:                      492
Model Family:                Binomial   Df Model:                            6
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -282.83
Date:                Mon, 02 Mar 2026   Deviance:                       565.65
Time:                        20:31:27   Pearson chi2:                     500.
No. Iterations:                     4   Pseudo R-squ. (CS):            0.06925
Covariance Type:            nonrobust                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

### Interpreting Coefficients

The coefficients represent the change in **log-odds** of the outcome for a one-unit change in the predictor, holding all other predictors constant.

Exponentiating the coefficients gives the **odds ratios**, which are easier to interpret.

In [4]:
# Odds ratios (excluding the intercept)
odds_ratios = np.exp(result.params.drop('Intercept'))
print(odds_ratios.round(3))

C(rx)[T.1.0 mg estrogen]    0.368
C(rx)[T.5.0 mg estrogen]    0.834
C(rx)[T.placebo]            0.964
age                         1.037
wt                          0.981
stage                       1.681
dtype: float64


### Analysis of Deviance

Since this is not a linear model, a standard ANOVA is not appropriate. Instead we use a **likelihood ratio test** (Type II), analogous to R's `car::Anova()` for GLMs.

In [5]:
# Likelihood ratio (Type II) tests for each predictor
# Compare the full model to models dropping each term one at a time
full_ll = result.llf

print(f"{'Predictor':<15} {'LR stat':>10} {'df':>5} {'p-value':>12}")
print('-' * 45)

for var in ['age', 'wt', 'stage', 'C(rx)']:
    # Fit reduced model without this predictor
    reduced_formula = 'death ~ ' + ' + '.join(v for v in ['age', 'wt', 'stage', 'C(rx)'] if v != var)
    reduced_result = smf.glm(
        reduced_formula,
        data=data_prostate,
        family=sm.families.Binomial()
    ).fit()

    lr_stat = 2 * (full_ll - reduced_result.llf)
    df = result.df_model - reduced_result.df_model
    p_value = stats.chi2.sf(lr_stat, df)
    label = var.replace('C(', '').replace(')', '')
    print(f"{label:<15} {lr_stat:>10.4f} {df:>5} {p_value:>12.6f}")

Predictor          LR stat    df      p-value
---------------------------------------------
age                 9.0638     1     0.002607
wt                 12.8776     1     0.000333
stage               6.0244     1     0.014110
rx                 17.3954     3     0.000586


The main effect of medication (`rx`) is very high and statistically significant.

**Note:** Be careful with causal interpretations of this model.